<a href="https://colab.research.google.com/github/najwahsbl/skin-expert-system/blob/main/ask_my_notes_rag_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ask My Notes - Build Your Own RAG Chatbot
### Unlocking LLM with AIS - Session 2 (Hands-on)
Artificial Intelligence Society (AIS) | FSKM, UiTM | *Evolusi Dekad 7*

Conductors: **Muhammad Fareed 'Aidil** & **Danial Syafiq** (AI Workflow Developer)

`#DiscoverThePresentOfFuture`

---

## What we are building

A **RAG chatbot** called **"Ask My Notes"** that answers questions from *your own* lecture notes, past-year exam papers, or a research-paper PDF.

RAG = **R**etrieval **A**ugmented **G**eneration (`penjanaan bantuan capaian`). Instead of hoping the LLM "already knows" your notes, we:

1. **Chunk** - break your notes into small pieces (`bahagikan`)
2. **Embed** - turn each piece into a vector / `embedding = perwakilan vektor`
3. **Retrieve** - find the pieces most relevant to a question (`capai`)
4. **Generate** - give those pieces to the LLM and ask it to answer (`jana jawapan`)
5. **Guardrail** - force it to answer ONLY from your notes, with citations (`pagar keselamatan`)

This is **exactly** the kind of feature you can put in your **Final Year Project (FYP)**.

---

## Everything here is FREE (no payment, no credit card)

| Part | Tool | Cost |
|------|------|------|
| LLM (answers) | Google **Gemini** API free tier (backup: **Groq**) | Free |
| Embeddings | `sentence-transformers` all-MiniLM-L6-v2 (runs in Colab) | Free |
| Vector DB | **ChromaDB** (in-memory, in Colab) | Free |

## Get your FREE Gemini API key (2 minutes)

1. Go to **https://aistudio.google.com/app/apikey**
2. Sign in with any Google account.
3. Click **"Create API key"** and copy it (it looks like `AIza...`).
4. Paste it when this notebook asks (Step 3 below).

*(Optional backup) Free Groq key: https://console.groq.com/keys*

---

### How to run

> Click **Runtime > Run all** (Larian > Jalankan semua), then scroll down and follow the prompts. Run cells one by one if you prefer.


## Step 1 - Install the libraries

We install everything we need. `-q` = quiet (less output). This takes about 1-2 minutes the first time. If you see a "restart runtime" button you can ignore it for this notebook.


In [ ]:
# Step 1: install free libraries (quiet)
# chromadb              -> our local vector database
# sentence-transformers -> free local embeddings (all-MiniLM-L6-v2)
# pypdf                 -> read text out of PDF files
# google-generativeai   -> talk to Gemini (free tier)         [optional key]
# groq                  -> backup free LLM provider           [optional key]
# transformers/accelerate -> run a small open model LOCALLY    [NO key needed]
# rank-bm25             -> keyword search for the OPTIONAL hybrid step
%pip install -q chromadb sentence-transformers pymupdf pypdf google-generativeai groq transformers accelerate rank-bm25
print("Done installing. You can move to Step 2.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.0/346.0 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7

## Step 2 - Import everything

If this cell runs with no red errors, your libraries installed correctly.


In [ ]:
# Step 2: imports
import os
import re
import textwrap

import chromadb
from sentence_transformers import SentenceTransformer

print("Imports OK. Ready for Step 3.")


Imports OK. Ready for Step 3.


## Step 3 - Set your API keys

We try to read keys from Colab's **Secrets** (the key icon on the left sidebar) first. If you have not added them there, the notebook will simply **ask you to paste** the key.

- `GEMINI_API_KEY` - **required** (free, from https://aistudio.google.com/app/apikey).
- `GROQ_API_KEY` - *optional* backup (free, from https://console.groq.com/keys). Just press Enter to skip.

> Tip for the workshop: pasting once here is fine. For real projects, use Colab Secrets so you never hard-code keys.


In [ ]:
# Step 3: load API keys from Colab Secrets, else ask the user to paste them.
from getpass import getpass


def _get_key(name, required=False):
    """Return an API key from Colab Secrets, environment, or a typed prompt."""
    # 1) Try Colab Secrets (the key icon in the left sidebar)
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            print(f"{name}: loaded from Colab Secrets.")
            return val.strip()
    except Exception:
        pass

    # 2) Try environment variable
    if os.environ.get(name):
        print(f"{name}: loaded from environment.")
        return os.environ[name].strip()

    # 3) Ask the user to paste it (hidden input)
    prompt = f"Paste {name}"
    if not required:
        prompt += " (optional - press Enter to skip)"
    val = getpass(prompt + ": ")
    # Colab quirk: getpass can return a dict {} instead of "" when Enter is hit
    val = "" if isinstance(val, dict) else str(val).strip()
    return val


GEMINI_API_KEY = _get_key("GEMINI_API_KEY", required=True)
GROQ_API_KEY = _get_key("GROQ_API_KEY", required=False)

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY or ""
os.environ["GROQ_API_KEY"] = GROQ_API_KEY or ""

if not GEMINI_API_KEY:
    print("\nWARNING: No Gemini key entered. Set PROVIDER = 'groq' in Step 4,")
    print("or re-run this cell and paste a free Gemini key.")
else:
    print("\nKeys ready. Go to Step 4.")

GROQ_API_KEY: loaded from Colab Secrets.

or re-run this cell and paste a free Gemini key.


## Step 4 - The PROVIDER SWITCH (choose your free LLM)

Pick how answers get generated. **All three are free.** Change the ONE line
`PROVIDER = "..."` in the next cell.

| PROVIDER  | Needs a key?            | Notes |
|-----------|------------------------|-------|
| `"gemini"`| Free Google AI Studio key | High quality, fast. Key from aistudio.google.com (no credit card). |
| `"groq"`  | Free Groq key          | Very fast, generous free limits. Key from console.groq.com. |
| `"local"` | **No key at all**      | Runs a small open model *inside* Colab. Zero signup, zero cost, no rate limits. Tip: turn on a GPU (**Runtime > Change runtime type > T4 GPU**); the first answer downloads the model (~1-3 min). |

> For RAG, even the tiny **local** model works surprisingly well, because it only
> has to *read the notes we retrieved* and answer - it doesn't need to "know" much.


In [ ]:
# Step 4: PROVIDER SWITCH  <-- change this one line to switch free providers
PROVIDER = "groq"   # options: "gemini"  |  "groq"  |  "local"  (no key needed)

# Cloud (free key) models:
GEMINI_MODEL = "gemini-2.5-flash"          # alt: "gemini-1.5-flash"
GROQ_MODEL = "llama-3.3-70b-versatile"
# If gemini-2.0-flash is rejected for your key/region, we auto-try these next:
GEMINI_FALLBACKS = ["gemini-2.5-flash", "gemini-flash-latest"]

# Local (NO key) model - small, free, runs inside Colab. Good for RAG.
LOCAL_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
_LOCAL_PIPE = None   # the local model is loaded once, on first use


def generate(prompt, temperature=0.2):
    """Send a text prompt to the chosen FREE provider and return the answer text.

    Low temperature (0.2) keeps answers factual and grounded - good for RAG.
    Errors are surfaced HONESTLY (we show the real message, not a guess).
    """
    global _LOCAL_PIPE
    try:
        if PROVIDER == "gemini":
            import google.generativeai as genai
            key = os.environ.get("GEMINI_API_KEY", "").strip()
            if not key:
                return "[No GEMINI_API_KEY set. Re-run Step 3, or set PROVIDER = 'local' (no key).]"
            genai.configure(api_key=key)
            last_err = None
            for model_name in [GEMINI_MODEL] + GEMINI_FALLBACKS:
                try:
                    model = genai.GenerativeModel(model_name)
                    resp = model.generate_content(
                        prompt, generation_config={"temperature": temperature})
                    return resp.text
                except Exception as inner:
                    last_err = inner
                    low = str(inner).lower()
                    # key/quota problems won't be fixed by trying another model
                    if ("api key" in low or "api_key" in low or "permission" in low
                            or "quota" in low or "429" in low or "resource_exhausted" in low):
                        break
                    continue
            raise last_err

        elif PROVIDER == "groq":
            from groq import Groq
            key = os.environ.get("GROQ_API_KEY", "").strip()
            if not key:
                return "[No GROQ_API_KEY set. Re-run Step 3, or set PROVIDER = 'local' (no key).]"
            client = Groq(api_key=key)
            resp = client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
            )
            return resp.choices[0].message.content

        elif PROVIDER == "local":
            # No API key. Runs a small open model in this Colab runtime.
            import torch
            from transformers import pipeline
            if _LOCAL_PIPE is None:
                use_gpu = torch.cuda.is_available()
                print(f"(Loading local model {LOCAL_MODEL} - first time downloads "
                      f"~1-3 GB, please wait...)")
                _LOCAL_PIPE = pipeline(
                    "text-generation",
                    model=LOCAL_MODEL,
                    torch_dtype=torch.float16 if use_gpu else torch.float32,
                    device=0 if use_gpu else -1,
                )
                if not use_gpu:
                    print("(No GPU detected - running on CPU = slower. Tip: "
                          "Runtime > Change runtime type > T4 GPU, then re-run.)")
            messages = [{"role": "user", "content": prompt}]
            out = _LOCAL_PIPE(messages, max_new_tokens=400, do_sample=False)
            gen = out[0]["generated_text"]
            # chat pipelines return the full message list; take the assistant turn
            if isinstance(gen, list):
                return gen[-1]["content"].strip()
            return str(gen).strip()

        # ----------------------------------------------------------------
        # OPTIONAL: Claude (Anthropic) - uncomment if YOU have a Claude key.
        # 1) %pip install -q anthropic
        # 2) add CLAUDE_API_KEY in Step 3, and set PROVIDER = "claude" above.
        # elif PROVIDER == "claude":
        #     import anthropic
        #     client = anthropic.Anthropic(api_key=os.environ.get("CLAUDE_API_KEY", ""))
        #     resp = client.messages.create(
        #         model="claude-3-5-haiku-latest",
        #         max_tokens=1024, temperature=temperature,
        #         messages=[{"role": "user", "content": prompt}],
        #     )
        #     return resp.content[0].text
        # ----------------------------------------------------------------

        else:
            return f"[Unknown PROVIDER '{PROVIDER}'. Use 'gemini', 'groq', or 'local'.]"

    except Exception as e:
        err = str(e)
        low = err.lower()
        # Show the REAL cause - a 400 is NOT a rate limit!
        if ("api key not valid" in low or "api_key_invalid" in low or "invalid api key" in low
                or "api key not found" in low or "401" in low or "unauthor" in low
                or "permission" in low):
            return ("[API KEY problem -> get a fresh free key (Step 3), or use PROVIDER = 'local' "
                    "(no key). Real error: " + err + "]")
        if ("429" in low or "quota" in low or "resource_exhausted" in low or "rate limit" in low):
            return ("[RATE LIMIT / quota -> wait ~1 min, switch provider, or use 'local'. "
                    "Real error: " + err + "]")
        if ("404" in low or "not found" in low or "is not supported" in low):
            return ("[MODEL problem -> model name may be unavailable. Real error: " + err + "]")
        return "[LLM error (" + str(PROVIDER) + "): " + err + "]"


# Quick smoke test (uses 1 tiny request from the chosen provider)
print("Provider:", PROVIDER)
print(generate("In one short sentence, what is RAG in AI?"))


Provider: groq
RAG (Retrieval-Augmented Generation) in AI is a technique that combines a retriever model to fetch relevant information with a generator model to produce text based on that information.


## Step 5 - Load your documents

Two ways:

1. **Upload your own PDF** (lecture notes, past-year paper, a research paper) when the upload box appears.
2. **Skip the upload** - we include a built-in **sample CS259 note** so the notebook still works even with zero uploads.

For every document we keep its **source filename** and **page number** as *metadata* so we can show **citations** later.


In [ ]:
# Step 5: load documents (PDF upload OR built-in sample notes)
from pypdf import PdfReader

# Each "document" is a dict: {"source": filename, "page": int, "text": str}
documents = []

# --- Built-in sample (so the notebook ALWAYS works) -----------------------
# A tiny slice of CS259-style notes. Replace by uploading your own PDF.
SAMPLE_NOTES = """
CS259 Diploma in Computer Science - Sample Lecture Notes

Topic 1: Data Structures.
A stack is a Last-In-First-Out (LIFO) data structure. The two main operations
are push (add an item to the top) and pop (remove the item from the top).
A queue is a First-In-First-Out (FIFO) data structure. Items are added at the
rear (enqueue) and removed from the front (dequeue).

Topic 2: Big-O Notation.
Big-O describes how the running time of an algorithm grows as the input size n
grows. Linear search is O(n). Binary search is O(log n) but the list must be
sorted first. Bubble sort is O(n^2) in the worst case.

Topic 3: Databases.
A primary key uniquely identifies each row in a table. A foreign key links one
table to the primary key of another table. Normalisation reduces data
redundancy by splitting data into related tables.

Topic 4: Final Year Project (FYP) tips.
Choose a focused problem. A RAG chatbot that answers questions from your own
notes is a strong, modern FYP idea. Always cite your sources in the report.
"""


def load_pdf(file_bytes, filename):
    """Read a PDF page-by-page with footer stripping.

    Uses PyMuPDF (fitz) for better text extraction from slide-deck PDFs.
    Footers (page numbers, email addresses, hashtags) are removed so they
    don't pollute the chunks or the embeddings.
    Falls back to pypdf if fitz is somehow unavailable.
    """
    import io
    try:
        import fitz  # PyMuPDF  (installed in Step 1)
        doc = fitz.open(stream=file_bytes, filetype="pdf")
        pages = []
        for page_num, page in enumerate(doc, 1):
            text = page.get_text("text")
            # Strip footer lines: bare page numbers, email addresses, hashtags
            lines = text.split("\n")
            clean = [
                ln for ln in lines
                if not re.search(r"^\s*page\s*\d+\s*$", ln, re.IGNORECASE)
                and not re.search(r"@[A-Za-z]", ln)        # email addresses
                and not re.search(r"#[A-Za-z]", ln)        # hashtags
                and not re.match(r"^\s*\d+\s*$", ln)    # lone page numbers
                and len(ln.strip()) > 3                     # ignore stub lines
            ]
            page_text = "\n".join(clean).strip()
            if page_text:
                pages.append({"source": filename, "page": page_num, "text": page_text})
        doc.close()
        return pages
    except ImportError:
        # Fallback: pypdf (less accurate for slide-deck PDFs)
        from pypdf import PdfReader
        reader = PdfReader(io.BytesIO(file_bytes))
        pages = []
        for i, page in enumerate(reader.pages):
            text = (page.extract_text() or "").strip()
            if text:
                pages.append({"source": filename, "page": i + 1, "text": text})
        return pages


# --- Try an upload (works on Google Colab) --------------------------------
uploaded_any = False
try:
    from google.colab import files
    print("Upload one or more PDFs now (or click 'Cancel' to use the sample).")
    uploaded = files.upload()
    for filename, content in uploaded.items():
        if filename.lower().endswith(".pdf"):
            docs = load_pdf(content, filename)
            documents.extend(docs)
            uploaded_any = uploaded_any or len(docs) > 0
            print(f"Loaded {len(docs)} page(s) from {filename}")
        else:
            # treat any non-PDF as plain text
            documents.append({"source": filename, "page": 1,
                               "text": content.decode("utf-8", errors="ignore")})
            uploaded_any = True
            print(f"Loaded text file {filename}")
except Exception as e:
    print(f"(No Colab upload available: {e})")

# --- Fall back to the sample if nothing usable was uploaded ---------------
if not uploaded_any:
    documents.append({"source": "sample_cs259_notes.txt", "page": 1,
                      "text": SAMPLE_NOTES.strip()})
    print("Using the built-in CS259 sample notes.")

print(f"\nTotal document pages loaded: {len(documents)}")


Upload one or more PDFs now (or click 'Cancel' to use the sample).


Saving cs259-llm-sample-notes.pdf to cs259-llm-sample-notes.pdf
Saving Workshop_Sample_Document.pdf to Workshop_Sample_Document.pdf
Loaded 5 page(s) from cs259-llm-sample-notes.pdf
Loaded 5 page(s) from Workshop_Sample_Document.pdf

Total document pages loaded: 10


## Step 6 - Chunking (split notes into small pieces)

LLMs and vector search work best on **small pieces** of text, not whole documents. We split each page into chunks of about 600 characters with a 100-character **overlap** (so a sentence cut at the edge still appears whole in the next chunk).

**Why this matters (Session-1 theory):** naive *blind* 500-token chunking can cut tables and sentences in half. In production you use **smart chunking** - semantic / section-aware / table-aware - so each chunk keeps its meaning. Our simple paragraph-aware splitter is a beginner-friendly first step in that direction.


In [ ]:
# Step 6: a simple, well-commented chunker
def chunk_text(text, chunk_size=600, overlap=100):
    """Split text into overlapping chunks, trying to break on paragraph/sentence
    boundaries so we do not cut words in half.
    """
    text = re.sub(r"\n{3,}", "\n\n", text.strip())  # tidy big gaps
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        piece = text[start:end]
        # If we are not at the very end, try to end on a clean boundary.
        if end < len(text):
            for sep in ("\n\n", ". ", "\n", " "):
                cut = piece.rfind(sep)
                if cut > chunk_size * 0.5:  # only if the cut is not too early
                    piece = piece[:cut + len(sep)]
                    break
        piece = piece.strip()
        if piece:
            chunks.append(piece)
        # move forward, keeping an overlap so context is not lost
        # Advance by (piece_length - overlap).
        # If the piece is shorter than the overlap we have reached the very end
        # of the text — break immediately instead of looping 1 char at a time
        # (which would create hundreds of near-identical tiny chunks).
        step = len(piece) - overlap
        if step <= 0:
            break
        start += step
    return chunks


# Build a flat list of chunks, each remembering its source + page.
chunk_texts = []
chunk_metas = []
for doc in documents:
    for c in chunk_text(doc["text"]):
        chunk_texts.append(c)
        chunk_metas.append({"source": doc["source"], "page": doc["page"]})

print(f"Created {len(chunk_texts)} chunks from {len(documents)} page(s).")
print("\nExample chunk:\n")
print(textwrap.fill(chunk_texts[0][:300], width=80))


Created 62 chunks from 10 page(s).

Example chunk:

CS259 - Introduction to LLMs & RAG (Lecture Notes) Diploma in Computer Science
(CS259) - Part 4 supplementary notes for "Unlocking LLM with AIS". Use these
notes as test data for your "Ask My Notes" RAG chatbot. Page 1 - What is an LLM?
A Large Language Model (LLM) is an AI model trained on huge amo


## Step 7 - Embeddings + ChromaDB (the searchable memory)

Now we turn each chunk into an **embedding** (`perwakilan vektor` - a list of numbers that captures meaning). Chunks with similar meaning end up with similar vectors.

We use the FREE local model **`all-MiniLM-L6-v2`** (downloads once, then runs in Colab - no API, no cost), and store everything in an in-memory **ChromaDB** collection together with the source/page metadata.

> *Alternative:* Gemini free embeddings (`text-embedding-004`). For this workshop the local model keeps everything offline and free.


In [ ]:
# Step 7: embed chunks and store them in ChromaDB

# 1) Load the free local embedding model (first run downloads ~80MB).
print("Loading embedding model all-MiniLM-L6-v2 ...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# 2) Create an in-memory Chroma client + a fresh collection.
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("ask_my_notes")  # start clean if re-run
except Exception:
    pass
collection = chroma_client.create_collection("ask_my_notes")

# 3) Embed all chunks (batch) and add them with metadata + ids.
print(f"Embedding {len(chunk_texts)} chunks ...")
embeddings = embedder.encode(chunk_texts, show_progress_bar=True).tolist()

collection.add(
    ids=[f"chunk-{i}" for i in range(len(chunk_texts))],
    documents=chunk_texts,
    embeddings=embeddings,
    metadatas=chunk_metas,
)

print(f"\nStored {collection.count()} chunks in ChromaDB. Ready to retrieve.")


Loading embedding model all-MiniLM-L6-v2 ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding 62 chunks ...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Stored 62 chunks in ChromaDB. Ready to retrieve.


## Step 8 - Retrieve (find the most relevant chunks)

When a question comes in, we embed the **question** the same way, then ask ChromaDB for the **top-k** closest chunks. These become the *context* we feed the LLM.


In [ ]:
# Step 8: semantic retrieval
def retrieve(question, k=3):
    """Return the top-k most relevant chunks for a question.

    Each result is a dict: {text, source, page, distance}.
    """
    q_emb = embedder.encode([question]).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=k)
    hits = []
    for text, meta, dist in zip(
        res["documents"][0], res["metadatas"][0], res["distances"][0]
    ):
        hits.append({
            "text": text,
            "source": meta.get("source", "?"),
            "page": meta.get("page", "?"),
            "distance": round(dist, 3),
        })
    return hits


# Quick test
for h in retrieve("What is a stack?"):
    print(f"[{h['source']} p.{h['page']} dist={h['distance']}]")
    print(textwrap.fill(h["text"][:200], width=80))
    print()


[cs259-llm-sample-notes.pdf p.3 dist=1.481]
a student asks a question, embed the question and find the most similar chunks
(semantic search). 5. Generate - send the question + retrieved chunks to the LLM
(Gemini/Groq) with the grounding instruc

[cs259-llm-sample-notes.pdf p.1 dist=1.604]
s an AI model trained on huge amounts of text so it can predict the next word
(token) in a sentence. By predicting one token at a time, it can write
paragraphs, answer questions, summarise documents,

[cs259-llm-sample-notes.pdf p.4 dist=1.641]
p (free tools):   - Embeddings: all-MiniLM-L6-v2 (local, free)   - Vector DB:
ChromaDB (local, free)   - LLM: Gemini gemini-2.0-flash (free) - switch to Groq
llama-3.3-70b if needed Walkthrough: 1. Th



## Step 9 - Grounded generation (answer ONLY from your notes)

This is the **guardrail** step. We build a prompt that:

1. Injects the retrieved chunks as **CONTEXT**.
2. Tells the LLM: *"Answer ONLY from the context. If it is not there, say you don't know."*
3. Asks it to keep things short and student-friendly.

Then we print the answer **with citations** (source filename + page) so you - and your FYP examiner - can trust where it came from. This is basic **hallucination control**.


In [ ]:
import os
import re
import textwrap

import chromadb
from sentence_transformers import SentenceTransformer

# Step 9: build a grounded prompt and generate a cited answer
def build_prompt(question, hits):
    """Combine the retrieved chunks + question into a grounded RAG prompt."""
    context_blocks = []
    for i, h in enumerate(hits, 1):
        context_blocks.append(
            f"""[Source {i}: {h['source']} (page {h['page']})]
{h['text']}"""
        )
    context = "\n\n".join(context_blocks)

    return f"""You are **Ask My Notes**, a friendly study assistant for UiTM CS259 students.

Your main role is to answer questions using ONLY the provided CONTEXT.

Guidelines:

* Be friendly and conversational.
* If the user greets you, responds casually, or asks for help using the chatbot, reply normally.
* For academic or course-related questions, use ONLY the information in the CONTEXT.
* Do not use outside knowledge when answering study questions.
* Keep answers clear and concise.
* If the answer is not found in the CONTEXT, reply exactly:

"I don't know based on your notes."

Examples:

* User: "Hi"
  Assistant: "Hi! I'm Ask My Notes. What would you like to learn today?"

* User: "Thank you"
  Assistant: "You're welcome! Feel free to ask another question."

* User: "What is machine learning?"
  Assistant: [Answer only from CONTEXT]

* User: "Who invented machine learning?"
  Assistant: "I don't know based on your notes."



CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""


# ── Normal ask() — semantic search + page-aware expansion ────────────────
# Uses ChromaDB vector search + pulls ALL chunks from matched pages.
# This means "step 7" works here too — once any chunk on that page is
# found semantically, the rest of the page comes along automatically.
# For even better exact-keyword matching, use ask_hybrid() in Level-up A.
def ask(question, k=8, show_sources=True):
    """Semantic RAG with page-aware expansion.

    Retrieves top-k chunks by vector similarity, then pulls in every
    other chunk from those same pages so no content is left behind.
    """
    # 1) Semantic retrieval
    initial = retrieve(question, k=k)

    # 2) Page-aware expansion — fetch ALL chunks from matched pages
    hit_pages = set((h["source"], h["page"]) for h in initial)
    expanded, seen = list(initial), set(h["text"][:80] for h in initial)
    for text, meta in zip(chunk_texts, chunk_metas):
        if (meta["source"], meta["page"]) in hit_pages:
            key = text[:80]
            if key not in seen:
                expanded.append({
                    "text": text,
                    "source": meta["source"],
                    "page": meta["page"],
                    "distance": None,
                })
                seen.add(key)
    hits = expanded

    prompt = build_prompt(question, hits)
    answer = generate(prompt)

    print("Q:", question)
    print("\nA:", answer.strip())
    if show_sources:
        print("\nSources:")
        seen_tags = set()
        for h in hits:
            tag = f"  - {h['source']} (page {h['page']})"
            if tag not in seen_tags:
                print(tag)
                seen_tags.add(tag)
    print("-" * 70)
    return answer


# Quick smoke test
_ = ask("what is step 7 in the RAG pipeline?")


Q: what is step 7 in the RAG pipeline?

A: Step 7 in the RAG pipeline is Hybrid Retrieval, which uses BOTH semantic search (vector similarity) AND keyword search (BM25/TF-IDF) to find relevant chunks.

Sources:
  - cs259-llm-sample-notes.pdf (page 3)
  - Workshop_Sample_Document.pdf (page 4)
  - cs259-llm-sample-notes.pdf (page 5)
  - cs259-llm-sample-notes.pdf (page 4)
  - Workshop_Sample_Document.pdf (page 3)
  - Workshop_Sample_Document.pdf (page 1)
----------------------------------------------------------------------


## Step 10 - Chat with your notes

Call `ask("your question")` as many times as you like. Below are example questions against the sample notes - try your own too!

Notice the **guardrail** working: ask something *not* in the notes and it should say *"I don't know based on your notes."*


In [ ]:
# Step 10: example questions — Normal (Semantic) Retrieval
# These use ask() which is semantic-only (ChromaDB vector search).
# For a full 10-step list or exact keyword matches, see Level-up A below.

_ = ask("how does RAG work?")
_ = ask("what is RAG?")
_ = ask("Who won the football World Cup in 2018?")  # NOT in the notes -> guardrail


Q: how does RAG work?

A: RAG works by first retrieving relevant text from your documents and then asking the LLM to generate an answer using that text. It's a technique that fetches relevant chunks of your own data and feeds them to the LLM as context, so answers are grounded in your documents. 

Here's a step-by-step explanation: 
1. You upload a document (e.g., a PDF).
2. The document is split into chunks and converted into vectors (embeddings).
3. When you ask a question, the system finds the most relevant chunks using vector similarity search.
4. The relevant chunks + your question are sent to the LLM.
5. The LLM generates an answer based ONLY on the provided context.

Sources:
  - cs259-llm-sample-notes.pdf (page 3)
  - cs259-llm-sample-notes.pdf (page 5)
  - Workshop_Sample_Document.pdf (page 3)
  - cs259-llm-sample-notes.pdf (page 4)
  - cs259-llm-sample-notes.pdf (page 2)
----------------------------------------------------------------------
Q: what is RAG?

A: RAG stands for 

In [ ]:
# Step 10b: interactive chat — Normal (Semantic) Retrieval
# ─────────────────────────────────────────────────────────────────────────
# This loop uses SEMANTIC search only (ChromaDB vector similarity).
# Great for meaning-based questions like "how does X work?" or "what is Y?"
#
# Limitation: may miss exact number/keyword matches (e.g. "step 7").
# For that, run Level-up A first and use the HYBRID loop at the bottom.
# ─────────────────────────────────────────────────────────────────────────

print("Normal (Semantic) Retrieval — type 'quit' to stop.")
while True:
    try:
        q = input("\nAsk your notes (or 'quit'): ").strip()
    except Exception:
        break
    if not q or q.lower() in ("quit", "exit", "q"):
        print("Bye!")
        break
    ask(q)


Normal (Semantic) Retrieval — type 'quit' to stop.

Ask your notes (or 'quit'): step 7 in RAG pipeline
Q: step 7 in RAG pipeline

A: Step 7 in the RAG pipeline is Hybrid Retrieval. It uses BOTH semantic search (vector similarity) AND keyword search (BM25/TF-IDF) to find relevant chunks. Semantic search finds conceptually related text, while keyword search finds exact matches. Hybrid retrieval is always better than either alone. Tools used for this step include LangChain EnsembleRetriever.

Sources:
  - cs259-llm-sample-notes.pdf (page 3)
  - cs259-llm-sample-notes.pdf (page 5)
  - Workshop_Sample_Document.pdf (page 4)
  - Workshop_Sample_Document.pdf (page 3)
  - cs259-llm-sample-notes.pdf (page 4)
  - Workshop_Sample_Document.pdf (page 1)
----------------------------------------------------------------------

Ask your notes (or 'quit'): How a basic RAG pipeline works
Q: How a basic RAG pipeline works

A: A basic RAG pipeline works in the following steps: 
1. Ingest - read your PDF / n

---
# OPTIONAL LEVEL-UP (you can skip all of this)

These cells show how to make your RAG *better* - the same ideas from the Session-1 carousels. Great for standing out in your FYP, but **not required** for a working chatbot.


## Level-up A — Hybrid Retrieval (Semantic + Keyword + Page-Aware)

**Why you need this:** Pure semantic search finds *meaning* but can miss exact terms
like `step 7`, a formula, a specific code, or any structured list where the middle
items don't score high enough on their own.

This level-up adds two improvements:

| Upgrade | What it does | Fixes |
|---------|-------------|-------|
| **Hybrid Retrieval** | Merges ChromaDB (semantic) + BM25 (keyword) | Exact word/number matches like "step 7" |
| **Page-Aware Expansion** | Fetches ALL chunks from any matched page | Missing middle chunks in long lists |

After running the cell below you get `ask_hybrid()` — a drop-in replacement for
`ask()` that works better for structured documents. Use `ask()` for quick
meaning-based questions, `ask_hybrid()` when you need precision or completeness.


In [ ]:
# Level-up A: Hybrid Retrieval + Page-Aware Expansion
# ─────────────────────────────────────────────────────────────────────────
# TWO upgrades combined:
#
#  1. HYBRID RETRIEVAL — semantic (ChromaDB) + keyword (BM25) merged.
#     Semantic search finds meaning. BM25 finds exact words/numbers.
#     Together they almost never miss a relevant chunk.
#
#  2. PAGE-AWARE EXPANSION — once a page is found relevant, ALL chunks
#     from that page are included. Solves "middle steps missing" for
#     long structured content like a 10-step list.
# ─────────────────────────────────────────────────────────────────────────

from rank_bm25 import BM25Okapi

# Build a BM25 keyword index over all chunks (whitespace tokenisation).
_tokenised = [c.lower().split() for c in chunk_texts]
bm25 = BM25Okapi(_tokenised)
print(f"BM25 index built over {len(chunk_texts)} chunks.")


def hybrid_retrieve(question, k=15):
    """Merge semantic (ChromaDB) and keyword (BM25) hits, de-duplicated.
    Returns up to k*2 results so multi-part content is not cut off.
    """
    # 1) Semantic hits
    semantic = retrieve(question, k=k)

    # 2) Keyword hits via BM25
    scores = bm25.get_scores(question.lower().split())
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    keyword = [
        {"text": chunk_texts[i], "source": chunk_metas[i]["source"],
         "page": chunk_metas[i]["page"], "distance": None}
        for i in top_idx if scores[i] > 0
    ]

    # 3) Merge: semantic first, then keyword, de-duplicate by first 80 chars
    merged, seen = [], set()
    for h in semantic + keyword:
        key = h["text"][:80]
        if key not in seen:
            merged.append(h)
            seen.add(key)

    return merged[: k * 2]   # allow up to k*2 results (not k+1)


def page_aware_retrieve(question, k=15):
    """Hybrid retrieve + pull ALL chunks from every matched page.

    Fixes the problem where middle chunks of a long section (like steps 2-8
    in a 10-step list) are not ranked high enough to appear in top-k alone.
    """
    initial = hybrid_retrieve(question, k=k)
    hit_pages = set((h["source"], h["page"]) for h in initial)

    expanded, seen = list(initial), set(h["text"][:80] for h in initial)
    for text, meta in zip(chunk_texts, chunk_metas):
        if (meta["source"], meta["page"]) in hit_pages:
            key = text[:80]
            if key not in seen:
                expanded.append({
                    "text": text, "source": meta["source"],
                    "page": meta["page"], "distance": None,
                })
                seen.add(key)

    return expanded


def ask_hybrid(question, k=15, show_sources=True):
    """Hybrid RAG: page-aware retrieve -> build prompt -> generate -> citations.
    Use this instead of ask() when you need exact keyword/number matches
    or complete multi-step answers from structured documents.
    """
    hits = page_aware_retrieve(question, k=k)
    prompt = build_prompt(question, hits)
    answer = generate(prompt)

    print("Q:", question)
    print("\nA:", answer.strip())
    if show_sources:
        print("\nSources:")
        seen = set()
        for h in hits:
            tag = f"  - {h['source']} (page {h['page']})"
            if tag not in seen:
                print(tag)
                seen.add(tag)
    print("-" * 70)
    return answer


# Quick test
_ = ask_hybrid("list all 10 steps in the RAG pipeline")


BM25 index built over 62 chunks.
Q: list all 10 steps in the RAG pipeline

A: The 10 steps in the RAG pipeline are:

1. Document Ingestion
2. OCR for Scanned PDFs
3. Table Extraction
4. Image & Chart Understanding
5. Smart Chunking
6. Embeddings + Vector Database
7. Hybrid Retrieval
8. Reranking
9. Answer Generation
10. Hallucination Control

Sources:
  - cs259-llm-sample-notes.pdf (page 3)
  - cs259-llm-sample-notes.pdf (page 5)
  - Workshop_Sample_Document.pdf (page 4)
  - Workshop_Sample_Document.pdf (page 3)
  - cs259-llm-sample-notes.pdf (page 4)
  - Workshop_Sample_Document.pdf (page 1)
  - cs259-llm-sample-notes.pdf (page 2)
  - Workshop_Sample_Document.pdf (page 2)
----------------------------------------------------------------------


In [ ]:
# Level-up A — Interactive Chat: Hybrid Retrieval
# ─────────────────────────────────────────────────────────────────────────
# Uses HYBRID search (semantic + BM25 keyword + page-aware expansion).
# Better than the normal loop for:
#   ✓ Numbered items  — "what is step 7?"
#   ✓ Exact terms     — specific codes, names, formulas
#   ✓ Long sections   — "list all 10 steps"
#   ✓ Any question    — hybrid is always >= semantic alone
# ─────────────────────────────────────────────────────────────────────────

print("Hybrid Retrieval — type 'quit' to stop.")
while True:
    try:
        q = input("\nAsk your notes (or 'quit'): ").strip()
    except Exception:
        break
    if not q or q.lower() in ("quit", "exit", "q"):
        print("Bye!")
        break
    ask_hybrid(q)


Hybrid Retrieval — type 'quit' to stop.

Ask your notes (or 'quit'): RAG vs Fine tuning
Q: RAG vs Fine tuning

A: RAG (Retrieval-Augmented Generation) and fine-tuning are two different approaches. Fine-tuning means re-training the LLM on new data, which can be expensive (costing thousands of USD) and slow (taking days to weeks). On the other hand, RAG adds knowledge at inference time, making it cheaper (costing cents per query), faster (taking seconds), and can be updated instantly by changing the document. For most real-world use cases, RAG is the right choice.

Sources:
  - cs259-llm-sample-notes.pdf (page 3)
  - Workshop_Sample_Document.pdf (page 3)
  - cs259-llm-sample-notes.pdf (page 5)
  - cs259-llm-sample-notes.pdf (page 4)
  - Workshop_Sample_Document.pdf (page 5)
  - Workshop_Sample_Document.pdf (page 1)
  - cs259-llm-sample-notes.pdf (page 2)
  - cs259-llm-sample-notes.pdf (page 1)
----------------------------------------------------------------------

Ask your notes (or 'qui

## Level-up B - Re-ranking (put the BEST chunk first)

The top embedding match is **not always** the best context (Carousel 2, step 8: *Reranking*). A cheap trick: re-order the retrieved chunks by how many **question keywords** they contain. For production you would use a proper **cross-encoder reranker** (e.g. `cross-encoder/ms-marco-MiniLM-L-6-v2` via `sentence-transformers`).


In [ ]:
# Level-up B: tiny keyword-overlap re-ranker
def rerank_by_overlap(question, hits):
    """Re-order chunks by how many question words appear in each chunk."""
    q_words = set(re.findall(r"\w+", question.lower()))

    def overlap(h):
        c_words = set(re.findall(r"\w+", h["text"].lower()))
        return len(q_words & c_words)

    return sorted(hits, key=overlap, reverse=True)


_q = "What are the main operations of a stack?"
_reranked = rerank_by_overlap(_q, retrieve(_q, k=3))
print("Re-ranked order (best first):")
for h in _reranked:
    print(f"  - {h['source']} p.{h['page']}: {h['text'][:60]}...")

# NOTE: for a real cross-encoder reranker:
# from sentence_transformers import CrossEncoder
# ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
# scores = ce.predict([(_q, h["text"]) for h in hits])


Re-ranked order (best first):
  - cs259-llm-sample-notes.pdf p.4: p (free tools):
  - Embeddings: all-MiniLM-L6-v2 (local, fre...
  - cs259-llm-sample-notes.pdf p.3: cument into smaller pieces so each piece fits in
the context...
  - cs259-llm-sample-notes.pdf p.3: a student asks a question, embed the question and find the m...


## Level-up C - A mini chat UI with Gradio (great for your FYP viva!)

Four lines give you a real chat box with a shareable link - perfect to **demo** during your FYP presentation.


In [ ]:
# Level-up C: Gradio mini-UI (run this cell, then use the chat box / link)
%pip install -q gradio
import gradio as gr


def chat_fn(message, history):
    # Use the same page-aware expansion as ask()
    initial = retrieve(message, k=8)

    hit_pages = set((h["source"], h["page"]) for h in initial)
    expanded, seen = list(initial), set(h["text"][:80] for h in initial)
    for text, meta in zip(chunk_texts, chunk_metas):
        if (meta["source"], meta["page"]) in hit_pages:
            key = text[:80]
            if key not in seen:
                expanded.append({"text": text, "source": meta["source"],
                                  "page": meta["page"], "distance": None})
                seen.add(key)

    answer = generate(build_prompt(message, expanded))
    cites = ", ".join(sorted({f"{h['source']} p.{h['page']}" for h in expanded}))
    return f"{answer.strip()}\n\nSources: {cites}"


gr.ChatInterface(
    chat_fn,
    title="Ask My Notes (AIS x UiTM)",
    type="messages"       # suppresses the deprecation warning too
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7714dc29a7091903bf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## You built a RAG chatbot! Now make it YOURS (FYP)

What you just built maps 1:1 to a real **Standard RAG** system (Carousel 1):
**Search -> Retrieve -> Answer**, with citations and a guardrail. Most real apps work great with Standard or Hybrid RAG - do **not** jump to Agentic/Graph RAG unless you truly need it.

### Ideas to adapt this for your Final Year Project

- **Swap the data:** upload *your* subject's notes, past-year papers, or a research PDF in Step 5.
- **Add memory:** save the Chroma collection to disk so it persists between sessions.
- **Go hybrid:** turn on Level-up A so exact terms (codes, names, formulas) are found.
- **Better PDFs:** for messy/scanned PDFs add OCR (Tesseract) and table extraction (Camelot) - see Carousel 2.
- **Ship a UI:** use the Gradio box (Level-up C) for your viva demo, or build an N8N no-code version (Track B).

### Keep these handy

- Free Gemini key: https://aistudio.google.com/app/apikey
- Free Groq key: https://console.groq.com/keys
- ChromaDB docs: https://docs.trychroma.com
- sentence-transformers: https://www.sbert.net
- Workshop handouts & slides: ask the AIS conductors (Fareed & Danial).

---

Thanks for joining **"Unlocking LLM with AIS"**! Tag us **@aisocietyuitm**  `#DiscoverThePresentOfFuture`
